# OmniVoice Project Studio — Google Colab (optimize branch)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/binhminhanh1235/OmniVoice/blob/optimize/notebooks/OmniVoice_Project_Studio_Colab.ipynb)

Performance-validation notebook for the `optimize` branch: Gradio Web UI + REST API + SSE jobs + MCP from one OmniVoice Studio server.

This notebook deliberately does not install or modify `master`.


In [ ]:
# Install the isolated optimization branch.
!pip install -q --upgrade "git+https://github.com/binhminhanh1235/OmniVoice.git@optimize"

import os
import torch
from omnivoice.hardware_quality import detect_hardware

# Keep transient model/runtime caches on the Colab VM rather than the Drive FUSE mount.
os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
os.environ.setdefault("TORCH_HOME", "/content/.cache/torch")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("CUDA:", torch.cuda.is_available())
hardware = detect_hardware()
print(hardware.summary())
for note in hardware.notes:
    print("-", note)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime: Runtime → Change runtime type → T4 GPU")


## Local-first workspace + Google Drive persistence

Generation now runs from `/content/OmniVoiceStudio` on the Colab VM instead of writing every checkpoint/WAV through the Google Drive mount. At startup, persisted data is restored from `MyDrive/OmniVoiceStudio`; while Studio is running, a background mirror syncs changed files back to Drive every 45 seconds and performs one final sync when the server exits.

This separates **fast execution storage** from **persistent storage**. A hard Colab runtime loss can lose at most the work since the most recent mirror interval; chunk/section resume still recovers everything already mirrored.


In [ ]:
from google.colab import drive
from pathlib import Path
import atexit
import shutil
import subprocess
import threading

drive.mount("/content/drive")
WORKSPACE = "/content/OmniVoiceStudio"
PERSISTENT_WORKSPACE = "/content/drive/MyDrive/OmniVoiceStudio"
SYNC_INTERVAL_SECONDS = 45
Path(WORKSPACE).mkdir(parents=True, exist_ok=True)
Path(PERSISTENT_WORKSPACE).mkdir(parents=True, exist_ok=True)

def _sync_tree(source, destination, *, delete=False):
    source_path = Path(source)
    destination_path = Path(destination)
    destination_path.mkdir(parents=True, exist_ok=True)
    rsync = shutil.which("rsync")
    if rsync:
        command = [
            rsync, "-a", "--exclude=*.tmp", "--exclude=.nfs*",
        ]
        if delete:
            command.append("--delete")
        command.extend([f"{source_path}/", f"{destination_path}/"])
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL)
        return
    shutil.copytree(source_path, destination_path, dirs_exist_ok=True)

# Restore once per notebook runtime. Re-running this cell does not overwrite newer local work.
if not globals().get("_OMNIVOICE_LOCAL_RESTORED", False):
    _sync_tree(PERSISTENT_WORKSPACE, WORKSPACE, delete=False)
    _OMNIVOICE_LOCAL_RESTORED = True
    print("Restored persisted Studio data to local runtime storage.")

def sync_workspace_to_drive():
    _sync_tree(WORKSPACE, PERSISTENT_WORKSPACE, delete=True)

# Stop an older mirror thread if this cell is re-run.
old_stop = globals().get("_OMNIVOICE_SYNC_STOP")
if old_stop is not None:
    old_stop.set()

_OMNIVOICE_SYNC_STOP = threading.Event()
def _mirror_loop():
    while not _OMNIVOICE_SYNC_STOP.wait(SYNC_INTERVAL_SECONDS):
        try:
            sync_workspace_to_drive()
        except Exception as exc:
            print("Workspace mirror warning:", type(exc).__name__, exc)

_OMNIVOICE_SYNC_THREAD = threading.Thread(
    target=_mirror_loop, name="omnivoice-drive-mirror", daemon=True
)
_OMNIVOICE_SYNC_THREAD.start()
atexit.register(sync_workspace_to_drive)

print("Execution workspace:", WORKSPACE)
print("Persistent mirror:", PERSISTENT_WORKSPACE)
print(f"Mirror interval: {SYNC_INTERVAL_SECONDS}s")


## Optional stable hostname + private access

Set `USE_STABLE_TUNNEL = True` only after creating a remotely-managed Cloudflare Tunnel pointing your hostname to `http://localhost:8000`.

Create these Colab Secrets:

- `CLOUDFLARE_TUNNEL_TOKEN`
- `OMNIVOICE_API_TOKEN`
- `OMNIVOICE_UI_USERNAME`
- `OMNIVOICE_UI_PASSWORD`

When enabled, the same hostname exposes `/ui`, `/api/v1`, `/mcp`, and `/health`.


In [ ]:
PUBLIC_URL = "https://omnivoice.example.com"
USE_STABLE_TUNNEL = False

if USE_STABLE_TUNNEL:
    from google.colab import userdata

    required_names = [
        "CLOUDFLARE_TUNNEL_TOKEN",
        "OMNIVOICE_API_TOKEN",
        "OMNIVOICE_UI_USERNAME",
        "OMNIVOICE_UI_PASSWORD",
    ]
    required = {}
    for name in required_names:
        try:
            required[name] = userdata.get(name)
        except Exception:
            required[name] = None
    missing = [name for name, value in required.items() if not value]
    if missing:
        raise RuntimeError("Missing Colab Secrets: " + ", ".join(missing))
    for name, value in required.items():
        os.environ[name] = value
    os.environ["OMNIVOICE_API_TOKEN_SCOPES"] = (
        "omnivoice:read,omnivoice:generate,omnivoice:queue,omnivoice:mcp"
    )
    os.environ["OMNIVOICE_PUBLIC_URL"] = PUBLIC_URL
    del required
    print("Stable private public URL configured:", PUBLIC_URL)


In [ ]:
if USE_STABLE_TUNNEL:
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod 700 /content/cloudflared
    !/content/cloudflared --version


## Launch

- Stable mode: unified server with Gradio `/ui`, REST, SSE, and MCP.
- Simple fallback: temporary Gradio share URL only.
- The Studio process always uses the local execution workspace; Drive mirroring stays outside the render hot path.


In [ ]:
try:
    if USE_STABLE_TUNNEL:
        !omnivoice-studio serve \
          --model k2-fsa/OmniVoice \
          --workspace "$WORKSPACE" \
          --asr-model openai/whisper-small.en \
          --asr-device cpu \
          --host 0.0.0.0 \
          --port 8000 \
          --tunnel \
          --cloudflared /content/cloudflared \
          --public-url "$OMNIVOICE_PUBLIC_URL"
    else:
        !omnivoice-project-studio \
          --model k2-fsa/OmniVoice \
          --workspace "$WORKSPACE" \
          --asr-model openai/whisper-small.en \
          --asr-device cpu \
          --share
finally:
    _OMNIVOICE_SYNC_STOP.set()
    sync_workspace_to_drive()
    print("Final Studio workspace sync completed.")
